In [2]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# === Config ===
NUM_SEQUENCES = 665
OUT_DIR = Path("../../../data/simulation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUTPUT_PATH = OUT_DIR / "engine_off.csv"
X_OUTPUT_PATH   = OUT_DIR / "engine_off_X.npy"
Y_OUTPUT_PATH   = OUT_DIR / "engine_off_y.npy"

FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]

# --- Simulators (as given) ---
def simulate_temp_pressure(state):
    temperature = []
    pressure = []
    if state == 0:  # Cold engine (off for hours)
        T = random.uniform(-5, 40)
        for _ in range(30):
            T += random.uniform(-0.07, 0.07)
            temperature.append(T)
            pressure.append(random.uniform(0.98, 1.02))
    elif state == 1:  # Recently turned off (cooling)
        T = random.uniform(95, 110)
        for _ in range(30):
            T += random.uniform(-0.5, -0.1)
            temperature.append(T)
            pressure.append(random.uniform(0.98, 1.02))
    return temperature, pressure, state

def simulate_rpm():
    return [0] * 30

def simulate_vibration():
    return [random.uniform(-0.001, 0.001) for _ in range(30)]

# --- Generate dataset ---
rows = []
X_list, y_list = [], []

for seq_num in range(NUM_SEQUENCES):
    state = random.choice([0, 1])
    temps, press, _ = simulate_temp_pressure(state)
    rpm_vals = simulate_rpm()
    vib_vals = simulate_vibration()
    label_str = "Engine Off (cold)" if state == 0 else "Engine Off (cooling)"

    for t in range(30):
        rows.append({
            "Time": t + 1,
            "Sequence": seq_num + 1,
            "Temperature": temps[t],
            "Pressure": press[t],
            "RPM": rpm_vals[t],
            "Vibration": vib_vals[t],
            "State": label_str
        })

    X_list.append(np.stack([temps, press, rpm_vals, vib_vals], axis=1))
    y_list.append(label_str)

df = pd.DataFrame(rows, columns=["Time","Sequence",*FEATURE_COLS,"State"])
X = np.array(X_list, dtype=np.float32)  # (N,30,4)
y = np.array(y_list)

df.to_csv(CSV_OUTPUT_PATH, index=False)
np.save(X_OUTPUT_PATH, X)
np.save(Y_OUTPUT_PATH, y)

print("✅ Engine Off dataset created.")
print("X:", X.shape, "| y:", y.shape)


✅ Engine Off dataset created.
X: (665, 30, 4) | y: (665,)
